# Étape 4 : amélioration de l'approche de classification

## Approche de modélisation
**Contexte métier :** un faux négatif (départ non détecté) coûte plus cher qu'un faux positif (geste de rétention inutile).
On cherche donc à améliorer le **recall de la classe 1** en gardant une **precision** acceptable.

**Modèles comparés :** régression logistique (avec `StandardScaler`), Random Forest et XGBoost, tous évalués de la même façon.

**Métriques :** recall, precision et F1 de la classe 1 au seuil de 0,5, et **PR-AUC** (`average_precision`) : l'aire sous la courbe précision-rappel. Elle mesure la **qualité du classement** des employés, indépendamment du seuil : on l'utilise pour comparer les modèles, puis on choisit le seuil. Pas d'accuracy (trompeuse avec 16 % de partis).

**Découpage :** train / test stratifié (80 / 20). **Le jeu de test n'est pas utilisé dans ce notebook** : il est réservé à l'évaluation finale (notebook 05). Les comparaisons se font par validation croisée stratifiée (5 plis) sur le train ; pour les petits écarts, par validation croisée répétée (5 plis × 5 répétitions = 25 découpages).

**Pistes d'amélioration, dans l'ordre :**
1. Pondération des classes
2. Choix du seuil de décision
3. Limiter l'overfitting (régularisation, profondeur des arbres)
4. Feature engineering
5. Sur-échantillonnage (SMOTE)

---
## 1. Chargement et séparation
Même séparation qu'à l'étape 3 : 80 % train / 20 % test, stratifiée (`stratify=y`, `random_state=42`). Les fonctions viennent de `preparation.py`.

In [9]:
from preparation import charger_donnees, preparer_donnees, ajouter_features
from sklearn.model_selection import train_test_split

X, y = preparer_donnees(charger_donnees())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

---
## 2. Banc d'essai commun
On évalue les trois modèles de la même façon, en validation croisée sur le train. La fonction `evaluer_modele` renvoie le recall, la precision, le F1 et la PR-AUC (moyennes sur les plis), ainsi que le recall sur les données d'apprentissage pour mesurer l'**overfitting**.

La régression logistique est placée dans un `Pipeline` avec `StandardScaler` : le scaler est ajusté pli par pli, sans fuite d'information vers le pli de validation.

In [10]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluer_modele(modele, nom):
    s = cross_validate(
        modele, X_train, y_train, cv=cv,
        scoring=["recall", "precision", "f1", "average_precision"],
        return_train_score=True,
    )
    return {
        "modèle": nom,
        "recall": s["test_recall"].mean(),
        "precision": s["test_precision"].mean(),
        "f1": s["test_f1"].mean(),
        "pr_auc": s["test_average_precision"].mean(),
        "recall train": s["train_recall"].mean(),
    }

modeles = {
    "Régression logistique": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss"),
}

resultats = pd.DataFrame([evaluer_modele(m, n) for n, m in modeles.items()]).round(3)
resultats

,modèle,recall,precision,f1,pr_auc,recall train
0,Régression logistique,0.437,0.768,0.548,0.646,0.479
1,Random Forest,0.216,0.825,0.332,0.555,1.000
2,XGBoost,0.363,0.696,0.469,0.589,1.000


**Conclusion :** sans réglage, la **régression logistique est la meilleure** (PR-AUC 0,646) et ne surapprend presque pas (recall 0,44 en validation contre 0,48 en train). Les deux modèles d'arbres apprennent le train par cœur (recall train = 1,00) : PR-AUC 0,555 pour le Random Forest et 0,589 pour XGBoost.

Pour les trois, la precision est élevée et le recall bas : avec 16 % de partis et un seuil de 0,5, les modèles ne signalent que les cas très sûrs.

---
## 3. Pondération des classes
`class_weight="balanced"` (régression logistique, Random Forest) et `scale_pos_weight` (XGBoost) comptent une erreur sur un parti environ 5 fois plus qu'une erreur sur un resté.

In [11]:
ratio = (y_train == 0).sum() / (y_train == 1).sum()

modeles_ponderes = {
    "Régression logistique (pondérée)": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
    "Random Forest (pondéré)": RandomForestClassifier(random_state=42, class_weight="balanced"),
    "XGBoost (pondéré)": XGBClassifier(random_state=42, eval_metric="logloss", scale_pos_weight=ratio),
}

resultats_ponderes = pd.DataFrame(
    [evaluer_modele(m, n) for n, m in modeles_ponderes.items()]
)

pd.concat([resultats, resultats_ponderes]).round(3).reset_index(drop=True)

,modèle,recall,precision,f1,pr_auc,recall train
0,Régression logistique,0.437,0.768,0.548,0.646,0.479
1,Random Forest,0.216,0.825,0.332,0.555,1.000
2,XGBoost,0.363,0.696,0.469,0.589,1.000
3,Régression logistique (pondérée),0.737,0.381,0.501,0.616,0.817
4,Random Forest (pondéré),0.379,0.630,0.470,0.544,1.000
5,XGBoost (pondéré),0.447,0.637,0.517,0.564,1.000


**Conclusion :** le recall monte nettement (régression logistique : 0,44 → 0,74) et la precision baisse (0,77 → 0,38). Mais la **PR-AUC ne progresse pas** : 0,646 → 0,616 (régression logistique), 0,555 → 0,544 (Random Forest), 0,589 → 0,564 (XGBoost).

La pondération ne rend donc pas le classement meilleur : elle déplace seulement le seuil de décision implicite. On préfère régler le seuil directement (section suivante).

---
## 4. Choix du seuil de décision
Par défaut, un employé est annoncé « parti » si sa probabilité dépasse 0,5. Ce seuil n'a rien d'obligatoire.

Pour comparer les modèles à égalité, on fixe le **recall à 0,75** et on regarde la precision obtenue. Les probabilités sont calculées **en validation croisée** (`cross_val_predict`) : chaque employé est prédit par un modèle qui ne l'a jamais vu.

*Fausses alertes par départ trouvé* = (1 − precision) / precision.

In [12]:
import numpy as np
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve

def precision_a_recall(proba, y, recall_cible=0.75):
    precision, recall, seuils = precision_recall_curve(y, proba)
    ok = np.where(recall[:-1] >= recall_cible)[0]
    i = ok[-1]
    return seuils[i], precision[i], recall[i]

for nom, m in modeles.items():
    proba = cross_val_predict(m, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    seuil, p, r = precision_a_recall(proba, y_train)
    print(f"{nom:22s} seuil {seuil:.2f} | recall {r:.2f} | precision {p:.2f} | "
          f"fausses alertes par départ trouvé {(1-p)/p:.1f}")

Régression logistique  seuil 0.17 | recall 0.75 | precision 0.40 | fausses alertes par départ trouvé 1.5
Random Forest          seuil 0.18 | recall 0.75 | precision 0.34 | fausses alertes par départ trouvé 1.9
XGBoost                seuil 0.03 | recall 0.75 | precision 0.38 | fausses alertes par départ trouvé 1.7


**Conclusion :** à recall égal (0,75), la régression logistique donne la meilleure precision (0,40, soit 1,5 fausse alerte par départ trouvé), devant XGBoost (0,38) et le Random Forest (0,34). Les écarts sont petits.

Le seuil d'XGBoost (0,03) est très bas : ses probabilités sont extrêmes parce qu'il apprend le train par cœur.

---
## 5. Limiter l'overfitting
- **Régression logistique :** régularisation plus forte (`C=0.1`).
- **Random Forest :** arbres moins profonds (`max_depth=5`, `min_samples_leaf=5`).
- **XGBoost :** arbres de profondeur 2, 200 arbres, `learning_rate=0.05`.

In [13]:
modeles_freines = {
    "Régression logistique (C=0.1)": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000, C=0.1)),
    "Random Forest (profondeur 5, feuilles de 5)": RandomForestClassifier(
        random_state=42, max_depth=5, min_samples_leaf=5),
    "XGBoost (profondeur 2, 200 arbres, lr 0.05)": XGBClassifier(
        random_state=42, eval_metric="logloss",
        max_depth=2, n_estimators=200, learning_rate=0.05),
}

resultats_freines = pd.DataFrame(
    [evaluer_modele(m, n) for n, m in modeles_freines.items()]
)

pd.concat([resultats.iloc[:3], resultats_freines]).round(3).reset_index(drop=True)

,modèle,recall,precision,f1,pr_auc,recall train
0,Régression logistique,0.437,0.768,0.548,0.646,0.479
1,Random Forest,0.216,0.825,0.332,0.555,1.000
2,XGBoost,0.363,0.696,0.469,0.589,1.000
3,Régression logistique (C=0.1),0.389,0.811,0.517,0.656,0.418
4,"Random Forest (profondeur 5, feuilles de 5)",0.121,0.894,0.202,0.552,0.191
5,"XGBoost (profondeur 2, 200 arbres, lr 0.05)",0.305,0.835,0.440,0.631,0.470


**Conclusion :**

| Modèle | PR-AUC avant | PR-AUC après | Recall train après |
|---|---|---|---|
| Régression logistique (`C=0.1`) | 0,646 | **0,656** | 0,42 |
| XGBoost (profondeur 2) | 0,589 | **0,631** | 0,47 |
| Random Forest (profondeur 5) | 0,555 | 0,552 | 0,19 |

XGBoost gagne le plus (+0,04) et ne mémorise plus le train (recall train 1,00 → 0,47). La régression logistique progresse un peu. Le **Random Forest ne gagne rien** en PR-AUC : son plafond ne vient pas de la profondeur.

Le recall au seuil de 0,5 reste bas pour tous : c'est le seuil qui est en cause, traité ci-dessous.

### Seuil après réglage
Même comparaison qu'en section 4 (recall fixé à 0,75), avec les modèles freinés.

In [14]:
for nom, m in modeles_freines.items():
    proba = cross_val_predict(m, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    seuil, p, r = precision_a_recall(proba, y_train)
    print(f"{nom:45s} seuil {seuil:.2f} | recall {r:.2f} | precision {p:.2f} | "
          f"fausses alertes par départ trouvé {(1-p)/p:.1f}")

Régression logistique (C=0.1)                 seuil 0.19 | recall 0.75 | precision 0.41 | fausses alertes par départ trouvé 1.5
Random Forest (profondeur 5, feuilles de 5)   seuil 0.16 | recall 0.75 | precision 0.37 | fausses alertes par départ trouvé 1.7
XGBoost (profondeur 2, 200 arbres, lr 0.05)   seuil 0.15 | recall 0.75 | precision 0.38 | fausses alertes par départ trouvé 1.6


**Conclusion :** à recall égal (0,75), la precision est de 0,41 pour la régression logistique, 0,38 pour XGBoost et 0,37 pour le Random Forest, soit **1,5 à 1,7 fausse alerte par départ trouvé**. Les trois modèles sont proches : les écarts sont dans le bruit de la validation croisée.

---
## 6. Feature engineering
Trois colonnes créées à partir de l'intuition métier (fonction `ajouter_features` dans `preparation.py`) :
- `revenu_par_experience` : revenu mensuel rapporté à l'expérience (sous-paiement relatif) ;
- `stagnation` : années depuis la dernière promotion rapportées à l'ancienneté ;
- `satisfaction_moyenne` : moyenne des 4 scores de satisfaction.

Les gains attendus sont petits. On compare donc avec et sans ces variables sur la PR-AUC, en **validation croisée répétée** (25 découpages), pour les deux modèles les plus prometteurs. Règle de décision : on garde les variables si le gain moyen est d'au moins +0,01 et si elles gagnent sur 20 découpages sur 25 ou plus.

In [15]:
from preparation import ajouter_features
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=7)
X_train_fe = ajouter_features(X_train.copy())

for nom in ["Régression logistique (C=0.1)",
            "XGBoost (profondeur 2, 200 arbres, lr 0.05)"]:
    m = modeles_freines[nom]
    avant = cross_val_score(m, X_train, y_train, cv=rcv,
                            scoring="average_precision", n_jobs=-1)
    apres = cross_val_score(m, X_train_fe, y_train, cv=rcv,
                            scoring="average_precision", n_jobs=-1)
    d = apres - avant
    print(f"{nom}\n  sans : {avant.mean():.3f} | avec : {apres.mean():.3f} | "
          f"écart moyen {d.mean():+.3f} | mieux sur {(d > 0).sum()} découpages sur {len(d)}\n")

Régression logistique (C=0.1)
  sans : 0.648 | avec : 0.658 | écart moyen +0.010 | mieux sur 21 découpages sur 25

XGBoost (profondeur 2, 200 arbres, lr 0.05)
  sans : 0.610 | avec : 0.611 | écart moyen +0.001 | mieux sur 14 découpages sur 25



**Conclusion :**
- **Régression logistique :** +0,010 de PR-AUC, meilleure sur 21 découpages sur 25 : on **garde** les variables. Un modèle linéaire ne peut pas construire seul ces rapports entre variables.
- **XGBoost :** +0,001, 14 découpages sur 25 : gain nul. Les arbres savent déjà combiner les variables.

On conserve les trois variables pour tous les modèles dans la suite : elles aident la régression logistique et ne gênent pas XGBoost. Le gain reste petit.

---
## 7. Sur-échantillonnage (SMOTE)
SMOTE fabrique des partis synthétiques, proches de vrais partis, pour rééquilibrer l'entraînement. Il est appliqué **uniquement sur les données d'entraînement de chaque pli**, grâce à un `Pipeline` d'`imblearn` (jamais sur le pli de validation, ce qui fausserait les scores).

In [16]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

smote_logreg = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("clf", LogisticRegression(max_iter=1000, C=0.1)),
])
smote_xgb = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("clf", XGBClassifier(random_state=42, eval_metric="logloss",
                          max_depth=2, n_estimators=200, learning_rate=0.05)),
])

reference = {"Régression logistique (C=0.1)": 0.658,
             "XGBoost (profondeur 2, 200 arbres, lr 0.05)": 0.611}

for nom, m in [("Régression logistique (C=0.1)", smote_logreg),
               ("XGBoost (profondeur 2, 200 arbres, lr 0.05)", smote_xgb)]:
    s = cross_val_score(m, X_train_fe, y_train, cv=rcv,
                        scoring="average_precision", n_jobs=-1)
    print(f"{nom}\n  sans SMOTE : {reference[nom]:.3f} | avec SMOTE : {s.mean():.3f} | "
          f"écart {s.mean() - reference[nom]:+.3f}\n")

Régression logistique (C=0.1)
  sans SMOTE : 0.658 | avec SMOTE : 0.636 | écart -0.022

XGBoost (profondeur 2, 200 arbres, lr 0.05)
  sans SMOTE : 0.611 | avec SMOTE : 0.479 | écart -0.132



**Conclusion :** SMOTE **dégrade** les deux modèles : −0,022 de PR-AUC pour la régression logistique et −0,132 pour XGBoost. Les partis synthétiques sont interpolés entre des employés réels : sur les colonnes 0 / 1 (heures supplémentaires, genre, poste…), ils prennent des valeurs irréalistes (par exemple 0,37) que les arbres apprennent à tort. On **rejette** SMOTE.

---
## 8. Synthèse de l'étape 4
Tous les résultats sont des validations croisées sur le train (le jeu de test n'a pas été utilisé).

| Test | Résultat | Décision |
|---|---|---|
| Modèles de base (PR-AUC) | régression logistique 0,646 ; XGBoost 0,589 ; Random Forest 0,555 | point de départ |
| Pondération des classes | PR-AUC inchangée ou en baisse | déplace seulement le seuil : non retenue |
| Limiter l'overfitting | régression logistique 0,656 ; XGBoost 0,631 ; Random Forest 0,552 | retenue (sauf pour le Random Forest, sans effet) |
| Seuil à recall 0,75 (precision) | 0,41 ; 0,38 ; 0,37 | le seuil est choisi à la fin, sur le train |
| Variables créées | régression logistique +0,010 ; XGBoost +0,001 | gardées |
| SMOTE | −0,022 ; −0,132 | rejeté |

- Les gains viennent surtout de la **régularisation** et du **choix du seuil**, pas du rééquilibrage des classes.
- La **régression logistique** et **XGBoost** sont à égalité statistique ; le **Random Forest** plafonne autour de 0,55 de PR-AUC et est écarté.
- À recall de 0,75, il faut compter environ 1,5 à 1,7 fausse alerte par départ trouvé.

**Prochaine étape (notebook 05) :** recherche d'hyperparamètres pour la régression logistique et XGBoost, choix du modèle final par validation croisée, seuil fixé sur le train, évaluation sur le test **une seule fois**, puis interprétation avec SHAP.